# Section 18 数据质量审计：可复算核对
## tl;dr
基于 3 份冻结审计 JSON，独立重算第 18.2 节阈值并核对逐类数量：NSL-KDD、CIC-2017、CIC-2018 为“保持”；UNSW-NB15、Malaya 为“去重重跑”。

Malaya 原缓存测试集有 3,780 / 10,370 行与训练集完全相同（36.4513%）。当前 Exp H 的 ID / shift 视图均为零重叠，但**不豁免数据集级去重重跑或全局冲突标签删除要求**。

本 notebook 仅加载审计统计，不运行训练，不使用 GPU，不做 oversampling，不修改数据，也不停止 Exp H。未将 CRR 的既有否决改写为待定。


## Context & Methods
目的：为第 18 节数据质量结论提供轻量、可复算的审计伴随文件，而不是重新扫描数百万原始行。

- 测试重叠率 = 与任一训练行具有完全相同特征的测试行数 / 全部测试行数；标签不参与匹配。
- 第 18.2 节：总体重叠率 ≥ 5%，或任一类重叠率 ≥ 20%，则“去重重跑”；否则“保持”。从整数分子、分母重算，不平均逐类比例。
- 逐类训练 / 测试行、测试匹配行、冲突标签受影响行可加总；跨类特征冲突意味着 unique 数、重复 excess 数、冲突组数不一定可加总，绝不把这些量直接相加。
- “保持”仅是重叠阈值判断，不表示数据无问题。全局冲突标签删除要求仍需单独执行；本审计没有清洗任何数据。

### Key Assumptions
输入统计来自对应审计脚本的精确特征比较。notebook 核对统计的一致性、来源 SHA256 和规则复算，但不独立重做原始数据逐行指纹。CIC 使用实际缓存的 feature-hash-group 划分；S1 是 2017 训练到 2018 偏移测试。KDDTest-21 是 KDDTest+ 的难样本子集，并非独立域。Malaya 是应用分类数据，不能泛称 NIDS 攻击分类。


## Data
### 1. 加载冻结证据
只使用仓库内 3 份报告 JSON；路径从仓库根或 notebook 目录自动向上解析。运行开始和结束都核对文件哈希，避免中途更新造成混合快照。所需运行时仅为 Python 标准库。


In [1]:
import hashlib
import json
import math
import platform
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "ncd/results/data_quality/section18").is_dir()
)
BASE = ROOT / "ncd/results/data_quality/section18"
FILES = {
    "tabular": BASE / "tabular/audit_tabular.json",
    "malaya": BASE / "malaya/audit_result.json",
    "cic": BASE / "cic/cic_audit.json",
}
payloads = {}
evidence = []
for name, path in FILES.items():
    raw = path.read_bytes()
    payloads[name] = json.loads(raw)
    evidence.append({
        "source": str(path.relative_to(BASE)).replace("\\", "/"),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "git_commit": payloads[name].get("git_commit"),
    })
tabular, malaya, cic = (payloads[k] for k in ("tabular", "malaya", "cic"))
assert {d["dataset"] for d in tabular["datasets"]} == {"NSL-KDD", "UNSW-NB15"}
assert {d["dataset"] for d in cic["datasets"]} == {"CIC-IDS-2017", "CSE-CIC-IDS2018"}
assert "exp_h_s1" in cic
print("Python:", platform.python_version())
for item in evidence:
    print(item["source"])
    print("  SHA256:", item["sha256"])
    print("  source commit:", item["git_commit"])


Python: 3.10.20
tabular/audit_tabular.json
  SHA256: 4ca955414614e2ae5cc7c6dcaf2edeea84614719bcfaf30fc736ae5b75104ebf
  source commit: ce48bfc647503d62027755c24a3f22247af35641
malaya/audit_result.json
  SHA256: 4c36e3faaacf5a103a9e88ee1ac27f082aa681b4f6da64d05635a3e51bba0ce0
  source commit: ce48bfc647503d62027755c24a3f22247af35641
cic/cic_audit.json
  SHA256: 01bbb1decbb022411334e435e1735dd8aa66c08c307248e495eb43153d329a11
  source commit: ce48bfc647503d62027755c24a3f22247af35641


### 2. 独立复算与可加总项核对
每类计数必须为非负整数，匹配行数不能超过测试行数。测试数为零的类不参与类阈值，且不伪造为“测得 0%”。以下代码不使用输入的 verdict 作为计算依据，只在复算后核对它。


In [2]:
views = []
checks = []

def table(headers, rows):
    strings = [[str(x) for x in row] for row in [headers, *rows]]
    widths = [max(len(row[i]) for row in strings) for i in range(len(headers))]
    for index, row in enumerate(strings):
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))
        if index == 0:
            print("-+-".join("-" * width for width in widths))

def check(condition, message):
    if not condition:
        raise AssertionError(message)
    checks.append(message)

def add_view(dataset, name, classes, total, train_key, test_key, match_key,
             rate_key, recorded_decision=None, conflict_key=None,
             conflict_total=None):
    check(len({r["class_name"] for r in classes}) == len(classes),
          f"{dataset}/{name}: unique class labels")
    normalized = []
    for row in classes:
        numbers = [row[k] for k in (train_key, test_key, match_key)]
        check(all(isinstance(n, int) and not isinstance(n, bool) and n >= 0
                  for n in numbers), f"{dataset}/{name}/{row['class_name']}: counts")
        train_n, test_n, match_n = numbers
        check(match_n <= test_n, f"{dataset}/{name}/{row['class_name']}: matches <= test")
        rate = match_n / test_n if test_n else None
        if test_n:
            check(math.isclose(rate, row[rate_key], rel_tol=1e-12, abs_tol=1e-12),
                  f"{dataset}/{name}/{row['class_name']}: class rate")
        normalized.append({
            "class_name": row["class_name"], "train": train_n, "test": test_n,
            "matched": match_n, "rate": rate,
        })
    sums = [sum(r[k] for r in normalized) for k in ("train", "test", "matched")]
    for key, actual in zip((train_key, test_key, match_key), sums):
        check(actual == total[key], f"{dataset}/{name}: class sum {key}")
    train_n, test_n, match_n = sums
    check(test_n > 0, f"{dataset}/{name}: nonempty test")
    if conflict_key is not None:
        check(sum(r[conflict_key] for r in classes) == conflict_total,
              f"{dataset}/{name}: conflict affected-row sum")
    rate = match_n / test_n
    check(math.isclose(rate, total[rate_key], rel_tol=1e-12, abs_tol=1e-12),
          f"{dataset}/{name}: overall rate")
    triggered = [r["class_name"] for r in normalized if r["rate"] is not None
                 and r["matched"] * 5 >= r["test"]]
    decision = "去重重跑" if match_n * 20 >= test_n or triggered else "保持"
    if recorded_decision is not None:
        check(decision == recorded_decision, f"{dataset}/{name}: recorded decision")
    views.append({
        "dataset": dataset, "view": name, "train": train_n, "test": test_n,
        "matched": match_n, "rate": rate,
        "max_class_rate": max(r["rate"] for r in normalized if r["rate"] is not None),
        "decision": decision, "trigger_classes": triggered, "classes": normalized,
    })

for dataset in tabular["datasets"]:
    total = dict(dataset["totals"])
    total["train_after_cap"] = total["train_rows"]
    total["test_after_cap"] = total["test_rows"]
    add_view(dataset["dataset"], "official", dataset["family_rows"], total,
             "train_after_cap", "test_after_cap", "test_matching_any_train_rows",
             "test_matching_any_train_rate", dataset["decision"],
             "model_family_combined_conflict_rows",
             total["conflicts"]["model_family"]["combined"]["affected_rows"])
    if "KDDTest21" in dataset:
        hard = dataset["KDDTest21"]
        train_by_class = {r["class_name"]: r["train_after_cap"]
                          for r in dataset["family_rows"]}
        rows = [dict(r, train_rows=train_by_class[r["class_name"]])
                for r in hard["family_rows"]]
        hard_total = {
            "train_rows": total["train_rows"], "test_rows": hard["actual_rows"],
            "test_matching_any_train_rows": hard["test_matching_any_train_rows"],
            "test_matching_any_train_rate": hard["test_matching_any_train_rate"],
        }
        add_view(dataset["dataset"], "KDDTest-21 subset", rows, hard_total,
                 "train_rows", "test_rows", "test_matching_any_train_rows",
                 "test_matching_any_train_rate")

for name, view in malaya["views"].items():
    add_view("Malaya", name, view["rows"], view["total"], "train_rows",
             "test_rows", "test_rows_matching_any_train", "test_overlap_rate",
             view["decision_18_2"], "union_label_conflict_train_rows",
             view["total"]["union_label_conflict_train_rows"])
    check(sum(r["union_label_conflict_test_rows"] for r in view["rows"])
          == view["total"]["union_label_conflict_test_rows"],
          f"Malaya/{name}: conflict test affected-row sum")

def add_cic_view(dataset, name, view):
    all_rows = [r for r in view["rows"] if r["class_name"] == "__all__"]
    check(len(all_rows) == 1, f"{dataset}/{name}: exactly one overall row")
    total = all_rows[0]
    rows = [r for r in view["rows"] if r["class_name"] != "__all__"]
    add_view(dataset, name, rows, total, "train_rows", "test_rows",
             "test_matches_any_train_rows", "test_matches_any_train_rate",
             view["decision"], "conflict_rows", total["conflict_rows"])

for dataset in cic["datasets"]:
    check(dataset["provenance"]["recomputed_split_mismatches"] == 0,
          dataset["dataset"] + ": reconstructed frozen split matches")
    for name, view in dataset["views"].items():
        add_cic_view(dataset["dataset"], name, view)
add_cic_view("CIC-IDS-2017", "ExpH S1 source ID", cic["exp_h_s1"]["source_id"])
add_cic_view("CSE-CIC-IDS2018", "ExpH S1 target (train=2017)",
             cic["exp_h_s1"]["target_shift"])

receipt = cic["input_receipts"]
receipt_path = ROOT / receipt["path"]
check(hashlib.sha256(receipt_path.read_bytes()).hexdigest() == receipt["sha256"],
      "CIC input-receipt file SHA256 matches audited result")
check(hashlib.sha256((ROOT / "ncd/audit_quality_cic.py").read_bytes()).hexdigest()
      == cic["audit_script_sha256"], "CIC current audit script matches source hash")
evidence.append({"source": str(receipt_path.relative_to(BASE)).replace("\\", "/"),
                 "sha256": receipt["sha256"], "git_commit": cic["git_commit"]})
print("Normalized views:", len(views))
print("Assertions passed so far:", len(checks))
print("CIC receipts and current audit-script source hashes: verified")


Normalized views: 14
Assertions passed so far: 472
CIC receipts and current audit-script source hashes: verified


## Results
### 3. 第 18.2 节数据集级判定
任一所列适用视图触发阈值，数据集级即“去重重跑”。原划分不因后续过滤视图零重叠而消失。表中 keep = 保持，dedup-rerun = 去重重跑；这不表示已执行清洗或重训。

第二张表明确列出测试重叠分子 / 分母及逐类最大值；raw / scaled 是原 float32 特征 / 实际训练变换后的矩阵。


In [3]:
dataset_order = ["NSL-KDD", "UNSW-NB15", "Malaya", "CIC-IDS-2017", "CSE-CIC-IDS2018"]
dataset_decisions = {}
decision_rows = []
for dataset in dataset_order:
    selected = [v for v in views if v["dataset"] == dataset]
    triggered = [v for v in selected if v["decision"] == "去重重跑"]
    decision = "去重重跑" if triggered else "保持"
    dataset_decisions[dataset] = decision
    decision_rows.append([dataset, len(selected),
                          "dedup-rerun" if triggered else "keep",
                          "; ".join(v["view"] for v in triggered) or "-"])
table(["Dataset", "Views", "18.2 decision", "Triggering view"], decision_rows)
check(dataset_decisions["Malaya"] == "去重重跑", "Malaya original split still governs")
print("\nMalaya 当前视图的保持不豁免该数据集全局冲突标签删除；本 notebook 未清洗或重训。")

view_aliases = {
    "original_cache_raw_train_vs_test": "original cache",
    "exp_h_raw_fit_vs_id": "ExpH raw fit/ID",
    "exp_h_raw_fit_vs_shift": "ExpH raw fit/shift",
    "exp_h_transformed_fit_vs_id": "ExpH scaled fit/ID",
    "exp_h_transformed_fit_vs_shift": "ExpH scaled fit/shift",
}
table(["Dataset", "View", "Train N", "Test N", "Matched N", "Overlap %", "Max class %"],
      [[v["dataset"], view_aliases.get(v["view"], v["view"]), v["train"], v["test"],
        v["matched"], f"{100*v['rate']:.4f}", f"{100*v['max_class_rate']:.4f}"]
       for v in views])


Dataset         | Views | 18.2 decision | Triggering view                 
----------------+-------+---------------+---------------------------------
NSL-KDD         | 2     | keep          | -                               
UNSW-NB15       | 1     | dedup-rerun   | official                        
Malaya          | 5     | dedup-rerun   | original_cache_raw_train_vs_test
CIC-IDS-2017    | 3     | keep          | -                               
CSE-CIC-IDS2018 | 3     | keep          | -                               

Malaya 当前视图的保持不豁免该数据集全局冲突标签删除；本 notebook 未清洗或重训。
Dataset         | View                        | Train N  | Test N  | Matched N | Overlap % | Max class %
----------------+-----------------------------+----------+---------+-----------+-----------+------------
NSL-KDD         | official                    | 125973   | 22544   | 664       | 2.9454    | 5.3697     
NSL-KDD         | KDDTest-21 subset           | 125973   | 11850   | 529       | 4.4641    | 6.4227     
UNSW-

### 5. 逐类数量总和、cap 与排除步骤
先核对每类数量与总体行数，再显示紧凑总表。Malaya 原训练集先分为 fit / ID，这不是 cap；shift 去除原训练集重叠行是单独的排除步骤。CIC 的 cap 前后计数直接来自 uncapped / ncd_capped 实际视图。跨标签冲突下，不对各类的 unique / duplicate excess / 冲突组数求和。


In [4]:
count_rows = []
for dataset in tabular["datasets"]:
    totals = dataset["totals"]
    for field in ("train_before_cap", "train_after_cap", "test_before_cap", "test_after_cap"):
        check(sum(r[field] for r in dataset["family_rows"]) == totals[field],
              f"{dataset['dataset']}: per-class sum {field}")
    count_rows.append([dataset["dataset"], totals["train_before_cap"],
                       totals["train_after_cap"], totals["test_before_cap"],
                       totals["test_after_cap"]])
for dataset in cic["datasets"]:
    before = next(r for r in dataset["views"]["uncapped"]["rows"]
                  if r["class_name"] == "__all__")
    after = next(r for r in dataset["views"]["ncd_capped"]["rows"]
                 if r["class_name"] == "__all__")
    check(after["train_rows"] <= before["train_rows"]
          and after["test_rows"] <= before["test_rows"],
          dataset["dataset"] + ": caps cannot increase rows")
    count_rows.append([dataset["dataset"], before["train_rows"], after["train_rows"],
                       before["test_rows"], after["test_rows"]])
counts = malaya["class_counts_and_handling"]
for row in counts:
    check(row["raw_original_train_rows"] == row["train_after_cap"] + row["id_after_cap"],
          row["class_name"] + ": original train = fit + ID")
    check(row["shift_before_overlap_exclusion"] == row["shift_overlap_excluded_rows"]
          + row["shift_after_overlap_exclusion"],
          row["class_name"] + ": shift exclusion reconciliation")
    for prefix in ("train", "id", "shift"):
        check(row[prefix + "_before_cap"] == row[prefix + "_after_cap"],
              row["class_name"] + ": no Malaya " + prefix + " cap")
fit_n = sum(r["train_after_cap"] for r in counts)
id_n = sum(r["id_after_cap"] for r in counts)
shift_before = sum(r["shift_before_overlap_exclusion"] for r in counts)
excluded_n = sum(r["shift_overlap_excluded_rows"] for r in counts)
shift_n = sum(r["shift_after_overlap_exclusion"] for r in counts)
malaya_original = next(v for v in views if v["dataset"] == "Malaya"
                       and v["view"] == "original_cache_raw_train_vs_test")
check(fit_n + id_n == malaya_original["train"], "Malaya original train sum")
check(shift_before == malaya_original["test"], "Malaya original shift sum")
check(excluded_n == malaya_original["matched"], "Malaya overlap excluded sum")
for name, expected in (("exp_h_raw_fit_vs_id", id_n),
                       ("exp_h_raw_fit_vs_shift", shift_n)):
    current = next(v for v in views if v["view"] == name)
    check(current["train"] == fit_n and current["test"] == expected,
          "Malaya actual prepared counts: " + name)
count_rows.append(["Malaya fit/ID", fit_n, fit_n, id_n, id_n])
table(["Dataset/view", "Train before", "Train after", "Test before", "Test after"],
      count_rows)
print(f"\nMalaya original train: {fit_n + id_n} = fit {fit_n} + ID {id_n}")
print(f"Malaya shift: {shift_before} - overlap exclusions {excluded_n} = {shift_n}")
print("All applicable additive per-class totals reconciled.")


Dataset/view    | Train before | Train after | Test before | Test after
----------------+--------------+-------------+-------------+-----------
NSL-KDD         | 125973       | 125973      | 22544       | 22544     
UNSW-NB15       | 175341       | 175341      | 82332       | 82332     
CIC-IDS-2017    | 2262203      | 610447      | 565673      | 146109    
CSE-CIC-IDS2018 | 12882081     | 240730      | 3255102     | 60198     
Malaya fit/ID   | 29098        | 29098       | 7274        | 7274      

Malaya original train: 36372 = fit 29098 + ID 7274
Malaya shift: 10370 - overlap exclusions 3780 = 6590
All applicable additive per-class totals reconciled.


### 6. 小样本与执行边界
下表显示模型类分区小于 100 的 NSL / UNSW 类、CIC 有原始标签精确 split 证据的小分区、以及 Malaya 小分区。CIC 原标签总数 ≥ 100 不代表各分区都 ≥ 100（例如 2018 XSS 的测试数为 44）。完整原始标签清单保留在输入报告，未以此紧凑表替代。没有 oversampling。


In [5]:
rare_rows = []
for dataset in tabular["datasets"]:
    for row in dataset["family_rows"]:
        train_n, test_n = row["train_after_cap"], row["test_after_cap"]
        if min(train_n, test_n) < 100:
            rare_rows.append([dataset["dataset"], row["class_name"],
                              row["task_index"], train_n, test_n, "original family"])
for dataset in cic["datasets"]:
    for row in dataset["raw_label_counts"]["rows"]:
        split = (row.get("rare_split_evidence") or {}).get("ncd_capped")
        if split is not None and min(split["train"], split["test"]) < 100:
            check(row.get("small_partition") is True,
                  dataset["dataset"] + "/" + row["raw_label"] + ": small partition flag")
            rare_rows.append([dataset["dataset"], row["raw_label"], row["task_index"],
                              split["train"], split["test"], row["mapped_family"]])
for row in counts:
    if row["small_partitions_lt_100"]:
        rare_rows.append(["Malaya", row["class_name"], row["task_index"],
                          row["train_after_cap"], row["shift_after_cap"],
                          "original application"])
table(["Dataset", "Label", "Task", "Train N", "Test/shift N", "Model mapping"], rare_rows)

for item in evidence:
    check(hashlib.sha256((BASE / item["source"]).read_bytes()).hexdigest() == item["sha256"],
          item["source"] + ": unchanged throughout notebook run")
check(len(dataset_decisions) == 5, "all five datasets have computed decisions")
print(f"\nPassed {len(checks)} consistency assertions across {len(views)} views.")
print("Validation: standard-Python ordered compile/exec only.")
print("Jupyter kernel and frontend / HTML rendering: NOT verified.")
print("No oversampling, data mutation, training, GPU use, or Exp H stop command.")


Dataset         | Label                      | Task | Train N | Test/shift N | Model mapping       
----------------+----------------------------+------+---------+--------------+---------------------
NSL-KDD         | U2R                        | 3    | 52      | 67           | original family     
UNSW-NB15       | Worms                      | 4    | 130     | 44           | original family     
CIC-IDS-2017    | Heartbleed                 | 0    | 10      | 1            | DoS                 
CIC-IDS-2017    | Infiltration               | 3    | 30      | 6            | Infiltration        
CIC-IDS-2017    | Web Attack – Sql Injection | 2    | 19      | 2            | WebAttack           
CSE-CIC-IDS2018 | Brute Force -XSS           | 2    | 186     | 44           | WebAttack           
CSE-CIC-IDS2018 | SQL Injection              | 2    | 66      | 21           | WebAttack           
Malaya          | TeamViewer                 | 3    | 608     | 37           | original application


## Takeaways
- UNSW-NB15（8,541 / 82,332）与 Malaya 原划分（3,780 / 10,370）触发 18.2；数据集级结论为去重重跑，尚未在本 notebook 中实施。
- NSL-KDD、CIC-2017、CIC-2018 未触发该重叠阈值。CIC S1 偏移视图为 46 / 60,198（0.0764%）；这只是重叠审计，不是模型有效性结果。
- Malaya 当前 Exp H fit/ID 与 fit/shift 的 raw 和 scaled 视图均为零重叠，不推翻原缓存问题，也不豁免全局冲突标签删除。
- 所有适用逐类可加总行数已核对。稀有类未 oversample；cap、过滤、原划分与当前视图保持分开。CRR 否决不变；审计没有停止现行 Exp H。

### 验证状态与复现
已用标准 Python 按 cell 顺序 compile / exec，保存 stdout，并检查 notebook JSON 基本结构、每类计数、比例、决策与输入快照哈希。仅检查了保存的文本输出；**没有通过 Jupyter kernel 执行，也没有 Jupyter 前端 / HTML 渲染验证**。当前环境未安装额外依赖。若环境已有 Jupyter，可从仓库根执行以下命令后检查生成页面：

    python -m jupyter nbconvert --execute --to notebook --inplace ncd/notebooks/section18_data_quality_audit.ipynb
    python -m jupyter nbconvert --to html ncd/notebooks/section18_data_quality_audit.ipynb

若命令报告缺少 Jupyter / nbconvert / ipykernel，应在获准且具备依赖的环境复现；本 notebook 不自动安装。
